In [1]:
from google.colab import files
files.upload()   # choose kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d sriramr/fruits-fresh-and-rotten-for-classification -p data_fresh --unzip

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/sriramr/fruits-fresh-and-rotten-for-classification
License(s): unknown
100% 3.58G/3.58G [00:23<00:00, 163MB/s] 



In [2]:
import os
for dp, dn, fn in os.walk("data_fresh"):
    if fn:
        print(dp, len(fn))

data_fresh/dataset/train/freshoranges 1466
data_fresh/dataset/train/rottenapples 2342
data_fresh/dataset/train/rottenoranges 1595
data_fresh/dataset/train/freshapples 1693
data_fresh/dataset/train/rottenbanana 2224
data_fresh/dataset/train/freshbanana 1581
data_fresh/dataset/dataset/train/freshoranges 1466
data_fresh/dataset/dataset/train/rottenapples 2342
data_fresh/dataset/dataset/train/rottenoranges 1595
data_fresh/dataset/dataset/train/freshapples 1693
data_fresh/dataset/dataset/train/rottenbanana 2224
data_fresh/dataset/dataset/train/freshbanana 1581
data_fresh/dataset/dataset/test/freshoranges 388
data_fresh/dataset/dataset/test/rottenapples 601
data_fresh/dataset/dataset/test/rottenoranges 403
data_fresh/dataset/dataset/test/freshapples 395
data_fresh/dataset/dataset/test/rottenbanana 530
data_fresh/dataset/dataset/test/freshbanana 381
data_fresh/dataset/test/freshoranges 388
data_fresh/dataset/test/rottenapples 601
data_fresh/dataset/test/rottenoranges 403
data_fresh/dataset/te

In [3]:
!pip -q install timm
import os, json, random, torch, timm
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm import tqdm

device = "cuda"

def find_dir(name):
    for dp, dn, fn in os.walk("data_fresh"):
        if os.path.basename(dp).lower() == name and dn:
            return dp

train_dir, test_dir = find_dir("train"), find_dir("test")
print(train_dir, test_dir)

mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
# No hue jitter on purpose: colour is the main signal for rotten vs fresh
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2),
    transforms.GaussianBlur(3, sigma=(0.1, 1.5)),
    transforms.ToTensor(), transforms.Normalize(mean, std)])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(mean, std)])

train_full = datasets.ImageFolder(train_dir, transform=train_tf)
val_full = datasets.ImageFolder(train_dir, transform=eval_tf)
test_ds = datasets.ImageFolder(test_dir, transform=eval_tf)
classes = train_full.classes
print(classes)

idx = list(range(len(train_full)))
random.seed(42)
random.shuffle(idx)
n_val = int(0.1 * len(idx))
train_ds = Subset(train_full, idx[n_val:])
val_ds = Subset(val_full, idx[:n_val])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=32, num_workers=2)
print(len(train_ds), len(val_ds), len(test_ds))

data_fresh/dataset/train data_fresh/dataset/dataset/test
['freshapples', 'freshbanana', 'freshoranges', 'rottenapples', 'rottenbanana', 'rottenoranges']
9811 1090 2698


In [4]:
model = timm.create_model("efficientnet_b0", pretrained=True,
                          num_classes=len(classes)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()
EPOCHS = 4

def evaluate(loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast("cuda"):
                pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total

best = 0
for epoch in range(EPOCHS):
    model.train()
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        with torch.autocast("cuda"):
            loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
    val_acc = evaluate(val_loader)
    print(f"Epoch {epoch+1}: val acc {val_acc:.3f}")
    if val_acc > best:
        best = val_acc
        torch.save(model.state_dict(), "freshness_effb0.pt")

model.load_state_dict(torch.load("freshness_effb0.pt"))
print("TEST accuracy:", round(evaluate(test_loader), 3))
json.dump(classes, open("freshness_classes.json", "w"))

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 1/4: 100%|██████████| 307/307 [01:57<00:00,  2.62it/s]


Epoch 1: val acc 0.994


Epoch 2/4: 100%|██████████| 307/307 [01:42<00:00,  3.00it/s]


Epoch 2: val acc 0.994


Epoch 3/4: 100%|██████████| 307/307 [01:42<00:00,  2.98it/s]


Epoch 3: val acc 0.996


Epoch 4/4: 100%|██████████| 307/307 [01:45<00:00,  2.90it/s]


Epoch 4: val acc 1.000
TEST accuracy: 0.998


In [6]:
from google.colab import files
files.download("freshness_effb0.pt")
files.download("freshness_classes.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>